[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramonzaca/MLSecOPs/blob/main/TP_03/03_model_and_data_monitoring_v3.ipynb)

**How's the model performing? - Practice 3**

*Your model from TP_01 is now served by the API from TP_02. Requests arrive every day. Two questions keep the HIC team awake: **is the model still right?** and **is someone messing with it?***

*Monitoring answers both, as far as statistics can. In this practice you'll use the [Evidently](https://github.com/evidentlyai/evidently) library to check the quality of incoming data, detect drift, and track the model's performance. Then you'll see what monitoring catches, and what it misses, when the data is manipulated on purpose.*

*We start with a small medical dataset to learn the tools. In the exercises you'll apply them to your own TP_01 model.*

# Setup

The versions are listed in `TP_03/requirements.txt`. On Colab, pip only adds what is missing.

**If pip upgraded or downgraded a package (e.g. Evidently), restart the kernel before running the next cell.** Python keeps the old version loaded until then, and the imports below fail with errors like `cannot import name 'DataDefinition' from 'evidently'`.

In [ ]:
import os

REQUIREMENTS_URL = "https://raw.githubusercontent.com/ramonzaca/MLSecOPs/main/TP_03/requirements.txt"
requirements = "requirements.txt" if os.path.isfile("requirements.txt") else REQUIREMENTS_URL
%pip install -q -r {requirements}

In [ ]:
import warnings

import numpy as np
import pandas as pd
from evidently import DataDefinition, Dataset, Regression, Report
from evidently.metrics import (
    MAE,
    DriftedColumnsCount,
    DuplicatedRowCount,
    MeanValue,
    MissingValueCount,
    OutRangeValueCount,
    QuantileValue,
    RowCount,
    ValueDrift,
)
from evidently.presets import DataDriftPreset, DataSummaryPreset, RegressionPreset
from evidently.tests import eq, gte, lt, lte
from sklearn import datasets
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Silence only the deprecation noise from Evidently's own dependencies.
# Don't silence *all* warnings: some of them (e.g. a scikit-learn version mismatch) matter.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"(evidently|litestar)(\.|$)")

> **Monitoring in three questions**
>
> Once a model is in production, nobody checks its answers one by one. Instead, we regularly compare a batch of recent data (the **current** data) with a batch we trust (the **reference** data) and ask:
>
> 1. **Data quality.** Is the data well-formed? No missing values, no duplicates, values in plausible ranges.
> 2. **Data drift.** Does the current data *look like* the reference data? If the population changes, a model trained on the old one may no longer be accurate.
> 3. **Model performance.** Are the predictions still good? This needs the true answers (*labels*), which usually arrive late: you know the real price of a house only when it sells. Until then, questions 1 and 2, and the drift of the **predictions** themselves, are the early warnings.

---

# The data and the model

The [Diabetes dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) describes 442 patients with 10 measurements (age, sex, body-mass index `bmi`, blood pressure `bp` and six blood tests `s1`–`s6`). The `target` is a measure of disease progression one year later. All features are already centred and scaled, which is why their values are small and close to 0.

We split it in **three** parts:

| Part | Size | Role |
|---|---|---|
| `train` | 50 % | trains the model |
| `reference` | 25 % | data the model has **never seen**, with its predictions: what "normal" looks like |
| `current` | 25 % | plays the role of this week's production data |

In [ ]:
data = datasets.load_diabetes(as_frame=True).frame
features = [col for col in data.columns if col != "target"]

train, holdout = train_test_split(data, test_size=0.5, random_state=42)
reference, current = train_test_split(holdout, test_size=0.5, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(train[features], train["target"])


def with_predictions(df):
    """Return a copy of df with the model's predictions in a `prediction` column."""
    return df.assign(prediction=model.predict(df[features]))


reference = with_predictions(reference)
current = with_predictions(current)

for name, df in [("train", with_predictions(train)), ("reference", reference), ("current", current)]:
    print(f"MAE on {name:9s}: {mean_absolute_error(df['target'], df['prediction']):5.1f}")

> **Why not use the training data as the reference?** Look at the numbers above. On the training data the model looks almost three times better than on any new data: it has partly **memorised** it (see TP_01). If the reference were the training data, the current data would always look like a big performance drop, and every alert would be a false alarm. The reference must be data the model has not been trained on.

Evidently needs to know the role of each column: which ones are features, which one is the target and which one is the prediction. We describe it once with a `DataDefinition`, and wrap each DataFrame in a `Dataset`. We also monitor the `prediction` column like a feature: its drift is an early warning when labels are not available yet.

In [ ]:
definition = DataDefinition(
    numerical_columns=features + ["prediction"],
    regression=[Regression(target="target", prediction="prediction")],
)


def as_dataset(df):
    return Dataset.from_pandas(df, data_definition=definition)


reference_ds = as_dataset(reference)
current_ds = as_dataset(current)

---

# 1. A first drift report

> **How is drift detected?** For each column, Evidently runs a statistical test that asks: *could these two samples come from the same distribution?* With small samples like ours, it uses the Kolmogorov–Smirnov test for numbers and a Z-test for categories. The test returns a **p-value**. A p-value below **0.05** means "very unlikely to be the same distribution", so the column is flagged as drifted. By default, the whole dataset is flagged when at least half of the columns drift.

In [ ]:
drift_report = Report([DataDriftPreset()])
drift_snapshot = drift_report.run(current_ds, reference_ds)  # note the order: current first
drift_snapshot

`reference` and `current` are two random halves of the same data, so we expect **no drift**, and that's what the report shows. Keep this in mind: in this setup nothing has changed yet. We'll make things change in sections 7 to 9.

---

# 2. Zooming in on one column

A report is a list of **metrics**. Instead of a preset, we can pick them one by one. Here: the drift, the mean and the first quartile of `bmi`.

In [ ]:
bmi_snapshot = Report(
    [
        ValueDrift(column="bmi"),
        MeanValue(column="bmi"),
        QuantileValue(column="bmi", quantile=0.25),
    ]
).run(current_ds, reference_ds)
bmi_snapshot

The same results are available as data, which is what a monitoring pipeline actually uses:

In [ ]:
for metric in bmi_snapshot.dict()["metrics"]:
    print(f"{metric['metric_name']:60s} {metric['value']:.4f}")

---

# 3. The same metric for many columns

Writing one line per column quickly gets tedious. A small helper builds the same metric for every feature:

In [ ]:
def metric_for_each_column(metric_class, columns=features, **parameters):
    return [metric_class(column=col, **parameters) for col in columns]


quantile_snapshot = Report(metric_for_each_column(QuantileValue, quantile=0.25)).run(current_ds, reference_ds)
quantile_snapshot

---

# 4. Data summary and model quality

Two more presets give a complete picture: `DataSummaryPreset` (statistics, missing values, duplicates… for each column) and `RegressionPreset` (error metrics and plots, using `target` and `prediction`).

In [ ]:
overview_snapshot = Report([DataSummaryPreset(), RegressionPreset()]).run(current_ds, reference_ds)
overview_snapshot

---

# 5. From reports to tests

A report is for a human. A **test** is for a machine: a metric plus a condition, with a pass/fail result. Tests are what let monitoring run every night and raise an alert without anyone reading plots.

The presets can add tests automatically with `include_tests=True`. When there is a reference, each automatic test checks that the current value is within **±10 % of the reference value**. Let's try it on our current data, which (we know) has not drifted at all:

In [ ]:
auto_snapshot = Report([DataSummaryPreset(), RegressionPreset()], include_tests=True).run(current_ds, reference_ds)

auto_tests = auto_snapshot.dict()["tests"]
auto_failed = [t["name"] for t in auto_tests if t["status"].name != "SUCCESS"]
print(f"{len(auto_failed)} of {len(auto_tests)} automatic tests fail on data that has NOT drifted. A few of them:")
for name in auto_failed[:8]:
    print("  -", name)

> **Alert fatigue.** Most of these failures are noise. With about 110 rows per batch, statistics naturally move by more than 10 % from one sample to the next. And "±10 % of a mean close to 0" is a band of almost zero width, so it fails every time. A monitoring system that raises dozens of false alerts every night gets muted, and then it misses the real one. In security operations this is a well-known failure mode.
>
> So we write our **own tests**, with thresholds we can justify.

Our monitoring suite, one condition per line:

In [ ]:
REFERENCE_MAE = mean_absolute_error(reference["target"], reference["prediction"])


def monitoring_suite():
    return [
        # Data quality
        RowCount(tests=[gte(30)]),  # enough rows for the statistics to mean something
        DuplicatedRowCount(tests=[eq(0)]),
        *metric_for_each_column(MissingValueCount, tests=[eq(0)]),
        # at most 5 % of values outside what the model saw during training
        *[
            OutRangeValueCount(column=col, left=train[col].min(), right=train[col].max(), share_tests=[lte(0.05)])
            for col in features
        ],
        # Data drift: fewer than 30 % of the features drift
        DriftedColumnsCount(columns=features, share_tests=[lt(0.3)]),
        # Prediction drift (available immediately, no labels needed); uses Evidently's default test
        ValueDrift(column="prediction"),
        # Performance (needs labels): the error must not grow by more than 20 %
        MAE(mean_tests=[lte(1.2 * REFERENCE_MAE)]),
    ]


def run_monitoring(current_df):
    return Report(monitoring_suite(), include_tests=True).run(as_dataset(current_df), reference_ds)


def failed_tests(snapshot):
    return [t for t in snapshot.dict()["tests"] if t["status"].name != "SUCCESS"]


suite_snapshot = run_monitoring(current)
print(f"{len(suite_snapshot.dict()['tests'])} tests, {len(failed_tests(suite_snapshot))} failed")
suite_snapshot

With explicit thresholds, clean data passes every test. Now let's see whether the suite reacts when something **does** change.

---

# 6. A monitoring gate

In production, the test results feed an automatic decision: raise an alert, block an automatic retraining, or stop the pipeline. Here is a minimal **gate** we'll reuse for every scenario:

In [ ]:
def monitoring_gate(current_df, name):
    snapshot = run_monitoring(current_df)
    failures = failed_tests(snapshot)
    status = "✅ PASS" if not failures else f"🚨 ALERT ({len(failures)} failed)"
    print(f"{name}: {len(current_df)} rows → {status}")
    for failure in failures:
        print(f"    - {failure['name']} → {failure['description']}")
    return snapshot


_ = monitoring_gate(current, "This week's data")

---

# 7. Scenario: the population changes

Nothing malicious here: the hospital opens a new clinic for older patients, and this week's data only contains patients older than the median. This is **natural drift**. The model was trained on a mixed population, so it may be less reliable on this one.

In [ ]:
older_patients = current[current["age"] > current["age"].median()]
_ = monitoring_gate(older_patients, "Older patients only")

Several features drift together (age is correlated with blood pressure and the blood tests), and the predictions drift too. The alert fires **before** any label is available, which is exactly what monitoring is for.

---

# 8. 🔐 Scenario: someone floods the API

The TP_02 API has no authentication and no rate limit. An attacker sends 25 copies of the same request with an absurd `bmi` value, to see how the model reacts or to skew whatever is computed from the traffic. Crude attacks like this are loud:

In [ ]:
bot_request = current.sample(1, random_state=0).assign(bmi=0.3)  # far outside the training range
flooded = with_predictions(pd.concat([current] + [bot_request] * 25).drop(columns="prediction"))
_ = monitoring_gate(flooded, "Current data + 25 bot requests")

Duplicates, out-of-range values and drift: three independent tests catch it.

---

# 9. 🔐 Scenario: a stealthy manipulation

A smarter attacker wants specific patients to get a **higher** predicted progression (say, to push them up a priority list). They change only two measurements (`bmi` and `s5`) of a few patients, and only to values that are **high but normal**: the 95th percentile of the reference data. Nothing is out of range, nothing is duplicated.

We try it on 5 %, 10 %, 20 % and 30 % of the patients:

In [ ]:
def manipulate(df, fraction, seed=1):
    """Set bmi and s5 to their (normal) 95th percentile for a random fraction of the rows."""
    tampered = df.drop(columns="prediction").copy()
    victims = tampered.sample(frac=fraction, random_state=seed).index
    for col in ["bmi", "s5"]:
        tampered.loc[victims, col] = reference[col].quantile(0.95)
    return with_predictions(tampered), victims


rows = []
for fraction in [0.05, 0.10, 0.20, 0.30]:
    tampered, victims = manipulate(current, fraction)
    failures = failed_tests(run_monitoring(tampered))
    rows.append(
        {
            "manipulated rows": f"{len(victims)} ({fraction:.0%})",
            "avg. prediction change on those rows": round(
                (tampered.loc[victims, "prediction"] - current.loc[victims, "prediction"]).mean(), 1
            ),
            "failed tests": len(failures),
            "which": "; ".join(f["name"] for f in failures) or "none: the attack goes unnoticed",
        }
    )
pd.DataFrame(rows)

Read the table from top to bottom. With a handful of rows, each targeted patient gets a large boost in predicted progression, and **all tests pass**. The attack only shows up once it touches enough rows to change the overall distribution of predictions, and the performance test fires only at the very end. Even then, it would fire only after the true outcomes are known, which may be months later.

> **🔐 What this means for security**
>
> - Monitoring is **statistical**. It detects changes in a population, not a single malicious input. A targeted manipulation of a few inputs stays below any reasonable threshold. This is the idea behind *evasion* attacks ([MITRE ATLAS AML.T0015 *Evade ML Model*](https://atlas.mitre.org/techniques/AML.T0015)), which TP_06 covers.
> - An attacker who knows your thresholds can stay just below them. Thresholds are security parameters: don't publish them.
> - Monitoring compares against the **reference** data. If an attacker can tamper with the reference, drift becomes invisible. Protect it like the training data (TP_01's checksums, TP_05's poisoning).
> - The defences stack up: per-request **input validation** (TP_02 v3), **monitoring** of batches (this TP), and **robust models** (TP_06). None of them is enough alone.

---

# 10. Saving the results

Snapshots can be saved as a standalone HTML page (for humans) or JSON (for dashboards and pipelines).

> **🔐 Reports are sensitive.** They contain statistics, distributions and plots of your data, and sometimes individual values. Store them with the same access rules as the data itself, and don't publish them.

In [ ]:
os.makedirs("reports", exist_ok=True)
suite_snapshot.save_html("reports/monitoring_suite.html")
suite_snapshot.save_json("reports/monitoring_suite.json")
print(sorted(os.listdir("reports")))

---

# Exercises

### Exercise 1: Monitor your own model

Replace the dataset and the model with the ones from TP_01 (the California housing data, and the model you exported as `TP_01_model.skops`). Upload the model file next to this notebook (on Colab: the *Files* panel on the left).

The cell below does the plumbing for you: it downloads the housing data (checking its hash), rebuilds TP_01's test set (same split, same seed), and loads your model safely. Then it's your turn:

- Which data is your **reference**? (Hint: it must be data the model has never been trained on.)
- There is no "production" data for housing. Build at least two **current** batches that simulate realistic changes, for example: only `INLAND` districts, or a few years later when incomes have grown by 15 %.
- Don't forget that `ocean_proximity` is a **categorical** column (`categorical_columns=` in the `DataDefinition`).

In [ ]:
# --- Exercise 1 plumbing: TP_01 data and model -------------------------------------------
import hashlib
import tarfile
import urllib.request

import skops.io as sio
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedShuffleSplit

HOUSING_URL = "https://github.com/ramonzaca/MLSecOPs/raw/main/datasets/housing/housing.tgz"
HOUSING_SHA256 = "d4cd501af90475f09b814c7447c7701f59bf28e8cf1180205ae5ace9737a0109"
MODEL_FILE = "TP_01_model.skops"


def sha256sum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


os.makedirs("datasets/housing", exist_ok=True)
if not os.path.isfile("datasets/housing/housing.csv"):
    urllib.request.urlretrieve(HOUSING_URL, "datasets/housing/housing.tgz")
    if sha256sum("datasets/housing/housing.tgz") != HOUSING_SHA256:
        raise ValueError("housing.tgz checksum mismatch. Refusing to use it.")
    with tarfile.open("datasets/housing/housing.tgz") as tgz:
        tgz.extractall(path="datasets/housing", filter="data")
housing = pd.read_csv("datasets/housing/housing.csv")

# Same stratified split and seed as TP_01 → same test set, never used for training
income_cat = pd.cut(housing["median_income"], bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1, 2, 3, 4, 5])
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, test_index = next(split.split(housing, income_cat))
housing_test = housing.iloc[test_index]


# The model refers to this class by name (see TP_01 and TP_02): it must exist before loading
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6


class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        return np.c_[X, rooms_per_household, population_per_household]


if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError(f"Upload {MODEL_FILE} (exported at the end of TP_01) next to this notebook.")
print("SHA-256 of the model:", sha256sum(MODEL_FILE), "→ compare it with the one printed in TP_01")
untrusted = sio.get_untrusted_types(file=MODEL_FILE)
assert set(untrusted) <= {"__main__.CombinedAttributesAdder", "numpy.dtype"}, f"unexpected types: {untrusted}"
housing_model = sio.load(MODEL_FILE, trusted=untrusted)

housing_features = [col for col in housing.columns if col != "median_house_value"]
print(f"{len(housing_test)} test districts, MAE of your model on them:",
      round(mean_absolute_error(housing_test["median_house_value"], housing_model.predict(housing_test[housing_features]))))

### Exercise 2: Build a monitoring gate for your model

Write a test suite for the housing model, like the one in section 5. Justify each threshold in a comment. Check that:

- your reference data against itself (or a random half of it) **passes**;
- each of your simulated batches from exercise 1 **fails**, and for the right reason.

How many automatic tests (`include_tests=True`) fail on the random half? What does that tell you?

### Exercise 3: The attacker's view

You are now the attacker. Find a manipulation of the housing test data that **raises the predicted price of at least 20 districts by more than $50,000** while **passing your test suite** from exercise 2.

Then switch sides: add a test that catches your attack, and check that clean data still passes. What does it cost you in false alarms?

### Exercise 4: Final report

Write a short report (in a markdown cell, or as an exported Evidently HTML report with your comments) with:

- what you found about your model and the housing data;
- which changes your monitoring detects, which it misses, and how fast;
- what you would add to protect the model in production (think of TP_02 as well).